In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# สร้าง Spark Session
spark = SparkSession.builder \
    .appName("Get Data") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 09:41:59 INFO SparkEnv: Registering MapOutputTracker
26/04/26 09:41:59 INFO SparkEnv: Registering BlockManagerMaster
26/04/26 09:41:59 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/26 09:41:59 INFO SparkEnv: Registering OutputCommitCoordinator


# Clean Comment Data

In [ ]:
# Read Title
col_post_selected = ['body',
                     'created_utc',
                     'is_submitter',
                     'author',
                     'removal_reason',
                     'subreddit',
                     'score',
                     'ups',
                     ]

paths = ["gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_Anthropic/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_Bard/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_ChatGPT/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_ChatGPTPro/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_ClaudeAI/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_DeepSeek/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_GeminiAI/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_OpenAI/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_grok/part*"]


comment = spark.read.parquet(*paths).select(col_post_selected)

In [ ]:
comment.cache()
comment.printSchema()

root
 |-- body: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- removal_reason: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- score: long (nullable = true)
 |-- ups: long (nullable = true)



In [ ]:
print(f"Number of rows is: {comment.count()}")

Number of rows is: 5753328


## Detect and Delete Unappropiate Text

In [ ]:
from pyspark.sql.functions import length, percentile_approx
from pyspark.sql import functions as F

# Calculate the length of the 'selftext' column, treating nulls as empty strings
comment_with_text_length = comment.withColumn("char_count", F.length(comment['body']))

# Calculate the 75th percentile of 'selftext_length' ยอมให้คลาดเคลื่อนได้ 5% แลกกับความเร็วและประหยัด Cost
p75 = comment_with_text_length.approxQuantile("char_count", [0.75], 0.05)[0]

# Drop rows where 'selftext_length' is greater than the 75th percentile
comment = comment_with_text_length.filter(F.col("char_count") <= p75).drop("char_count")

print(f"75th percentile of selftext length: {p75}")
print(f"Number of rows after filtering: {comment.count()}")

75th percentile of selftext length: 211.0


Number of rows after filtering: 4253874


In [ ]:
from pyspark.sql import functions as F

# Detect unappropiate text like spam (www.), gif or removed/deleted and replace by ""
# รวมทุก Pattern ไว้ในตัวแปรเดียว
# 1. https:[^\s]+  -> ลบ URL
# 2. !\[gif\][^\s]* -> ลบ gif และข้อความที่ติดกันไปจนเจอ space
# 3. \[removed\]    -> ลบ [removed]
# 4. \[deleted\]    -> ลบ [deleted]
combined_pattern = r"https:[^\s]+|!\[gif\][^\s]*|\[removed\]|\[deleted\]"

# Clean ข้อมูลในขั้นตอนเดียว
comment = comment.withColumn("body",
    F.trim(F.regexp_replace(F.col("body"), combined_pattern, ""))
)

# Delect Null row when body after clean is null
comment = comment.dropna(how='any', subset = 'body')

# Filter out rows where 'body' has 1 character these text don't have meaning at all
comment = comment.filter(F.length(F.col("body")) > 1)

print(f'After Drop missing and unappropiate text from body: {comment.count()}')

After Drop missing and unappropiate text from body: 3830715


## Convert time in epoch format to Date Time

In [ ]:
# Or using from_unixtime for a specific string format
comment = comment.withColumn("created_utc", F.from_unixtime(F.col("created_utc"), "yyyy-MM-dd HH:mm:ss"))
print("Convert Epoch to Data Time Task is Complete")

Convert Epoch to Data Time Task is Complete


## Create Column for Taging AI Service

In [ ]:
# Convert all title to lower case for appropiate matching with nest step
comment = comment.withColumn("body", F.lower(F.col("body")))
print("Convert Title to Lower Case")

Convert Title to Lower Case


In [ ]:
# Create ChatGPT_Tag
comment = comment.withColumn("ChatGPT_Tag",
                        F.when(
                            (F.col("body").contains("chatgpt")) |
                            (F.col("body").contains("openai")) |
                            (F.col("body").contains(" gpt")) | # เพิ่ม space ข้างหน้าเพื่อเลี่ยงคำอื่นที่สะกดคล้ายกัน
                            (F.col("body").contains("dall-e")) | # เพิ่ม DALL-E เพราะเป็น Product ของ OpenAI
                            (F.col("body").contains("sora")),    # เพิ่ม Sora
                            1
                              ).otherwise(0)
                      )

print("Create ChatGPT_Tag Complete")

# Create ClaudeAI_Tag
comment = comment.withColumn("ClaudeAI_Tag",
                        F.when(
                            (F.col("body").contains("claude")) |
                            (F.col("body").contains("anthropic")),
                            1
                        ).otherwise(0)
                      )

print("Create ClaudeAI_Tag Complete")

# Create Gemini_Tag
comment = comment.withColumn("GeminiAI_Tag",
                        F.when(
                            (F.col("body").contains("gemini")) |
                            (F.col("body").contains("bard")) |
                            (F.col("body").contains("google ai")) |
                            (F.col("body").contains("palm")), # Model พื้นฐานของ Google
                            1
                              ).otherwise(0)
                      )

print("Create GeminiAI_Tag Complete")

# Create DeepSeek_Tag
comment = comment.withColumn("DeepSeek_Tag",
                        F.when(
                            (F.col("body").contains("deepseek")) |
                            (F.col("body").contains("deep seek")), # บางคนพิมพ์แยกคำ
                            1
                              ).otherwise(0)
                      )

print("Create DeepSeek_Tag Complete")

# Create Grok_Tag
comment = comment.withColumn("Grok_Tag",
                        F.when(
                            (F.col("body").contains("grok")) |
                            (F.col("body").contains(" xai")), # ชื่อบริษัทของ Elon Musk ที่ทำ Grok
                            1
                        ).otherwise(0)
                      )

print("Create Grok_Tag Complete")

Create ChatGPT_Tag Complete
Create ClaudeAI_Tag Complete
Create GeminiAI_Tag Complete
Create DeepSeek_Tag Complete
Create Grok_Tag Complete


## Split Post by Tag Count

In [ ]:
# Create Tag_Count

comment = comment.withColumn("Tag_Count",
                       F.col("ChatGPT_Tag") +
                       F.col("ClaudeAI_Tag") +
                       F.col("GeminiAI_Tag") +
                       F.col("DeepSeek_Tag") +
                       F.col("Grok_Tag"))

print("Create Tag_Count Complete")

Create Tag_Count Complete


In [ ]:
comment.groupBy("Tag_Count").count().show()

+---------+-------+
|Tag_Count|  count|
+---------+-------+
|        1| 427962|
|        3|   4184|
|        4|    687|
|        2|  30604|
|        0|3367218|
|        5|     60|
+---------+-------+



In [ ]:
# Split DataSet by Tag_Count
# Dataset where there is 0 or 1 AI mentioned
comment_single_tag_or_none = comment.filter(F.col("Tag_Count") <= 1)

print(f"Total Rows: {comment.count()}")

comment_single_tag_or_none.cache()

print(f"Single/None Tag Rows: {comment_single_tag_or_none.count()}")



# Dataset where multiple AI products are mentioned (Mixed content)
comment_multi_tag = comment.filter(F.col("Tag_Count") > 1)

print(f"Multi-Tag Rows: {comment_multi_tag.count()}")

Total Rows: 3830715


Single/None Tag Rows: 3795180


Multi-Tag Rows: 35535


In [ ]:
# Save post_multi_tag on data lake
target_path_parquet = "reddit-ai-2/process_data/Comment_multi_tag/comment_multi_tag.parquet"

comment_multi_tag.write.mode("overwrite").parquet(target_path_parquet)

print(f"บันทึกไฟล์ Parquet ไปที่ {target_path_parquet} เรียบร้อยแล้ว")

comment.unpersist()
comment_multi_tag.unpersist()

26/04/26 09:52:18 WARN TaskSetManager: Lost task 12.0 in stage 28.0 (TID 294) (cluster-6880017126-bigdata-w-1.asia-southeast1-c.c.bigdata-reddit-491813.internal executor 1): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://cluster-6880017126-bigdata-m/user/root/reddit-ai-2/process_data/Comment_multi_tag/comment_multi_tag.parquet.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:775)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:494)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:893)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:893)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoin

26/04/26 09:53:47 WARN TaskSetManager: Lost task 5.0 in stage 29.0 (TID 305) (cluster-6880017126-bigdata-w-1.asia-southeast1-c.c.bigdata-reddit-491813.internal executor 1): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://cluster-6880017126-bigdata-m/user/root/reddit-ai-2/process_data/Comment_multi_tag/comment_multi_tag.parquet.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:775)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:494)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:893)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:893)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint

26/04/26 09:53:58 WARN TaskSetManager: Lost task 12.0 in stage 29.0 (TID 309) (cluster-6880017126-bigdata-w-1.asia-southeast1-c.c.bigdata-reddit-491813.internal executor 3): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://cluster-6880017126-bigdata-m/user/root/reddit-ai-2/process_data/Comment_multi_tag/comment_multi_tag.parquet.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:775)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:494)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:893)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:893)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoin

บันทึกไฟล์ Parquet ไปที่ reddit-ai-2/process_data/Comment_multi_tag/comment_multi_tag.parquet เรียบร้อยแล้ว


DataFrame[body: string, created_utc: string, is_submitter: boolean, author: string, removal_reason: string, subreddit: string, score: bigint, ups: bigint, ChatGPT_Tag: int, ClaudeAI_Tag: int, GeminiAI_Tag: int, DeepSeek_Tag: int, Grok_Tag: int, Tag_Count: int]

## Transform Multi Tag to a categorical tag

In [ ]:
# Transform multi-columns for tagging to a categorical AI tag for ease to use.
from pyspark.sql.types import StringType

def subreddit_mapping(subreddit):
  if subreddit in ['ChatGPT','ChatGPTPro','OpenAI']:
    return "ChatGPT"
  elif subreddit in ['ClaudeAI','Anthropic']:
    return "Claude"
  elif subreddit in ['GeminiAI','Bard']:
    return "Gemini"
  elif subreddit in ['DeepSeek']:
    return "DeepSeek"
  elif subreddit in ['grok']:
    return "Grok"
  else:
    return "Cann't Identify"

def tag_mapping(tag1,tag2,tag3,tag4,tag5):
  if tag1 == 1:
    return "ChatGPT"
  elif tag2 == 1:
    return "Claude"
  elif tag3 == 1:
    return "Gemini"
  elif tag4 == 1:
    return "DeepSeek"
  elif tag5 == 1:
    return "Grok"
  else:
    return "Cann't Identify"

# Assign to Spark by UDF
subreddit_mapping_udf = F.udf(subreddit_mapping, StringType())
tag_mapping_udf = F.udf(tag_mapping, StringType())

In [ ]:
comment_single_tag_or_none = comment_single_tag_or_none.withColumn("AI_Name",
    F.when(
        F.col("Tag_Count") == 0,
        subreddit_mapping_udf(F.col("subreddit")) # Case 1: ใช้ค่าจาก column 'reddit' (สมมติว่าชื่อ 'subreddit')
    ).when(
        F.col("Tag_Count") == 1,
        tag_mapping_udf(
            F.col("ChatGPT_Tag"),
            F.col("ClaudeAI_Tag"),
            F.col("GeminiAI_Tag"),
            F.col("DeepSeek_Tag"),
            F.col("Grok_Tag")
        ) # Case 2: ใส่ Input 5 ตัวตาม Tag
    ).otherwise("Multi-Tag or Other") # กรณี Tag_Count > 1
)

print("Create AI Name Task is Complete")

Create AI Name Task is Complete


In [ ]:
comment_single_tag_or_none.groupBy("AI_Name").count().show()

+--------+-------+
| AI_Name|  count|
+--------+-------+
|DeepSeek|  63319|
| ChatGPT|2801153|
|  Gemini| 296135|
|  Claude| 401455|
|    Grok| 233118|
+--------+-------+



In [ ]:
# Delete Insufficient feature.
comment_single_tag_or_none = comment_single_tag_or_none.drop(
    "ChatGPT_Tag",
    "ClaudeAI_Tag",
    "GeminiAI_Tag",
    "DeepSeek_Tag",
    "Grok_Tag",
    "Tag_Count"
)

print("Delete Insufficient Feature")

Delete Insufficient Feature


## According to 'post_single_tag_or_none' Split to 2 dataset
1. Sampling 50,000 rows for taged sentiment by LLM and Train Model for predict sentiment for the rest post dataset
2. The Rest rows


In [ ]:
from pyspark.sql import functions as F

# 1. Add a unique ID to each row so we can track what was selected
comment_single_tag_or_none = comment_single_tag_or_none.withColumn("row_id", F.monotonically_increasing_id())

# 2. Calculate the fraction needed to get approximately 100,000 rows
total_count = comment_single_tag_or_none.count()
fraction = 50000 / total_count

# 3. Create the sampling (Selected data)
# Note: sample() is probabilistic, so we use limit(100000) to get exactly that amount
sampling = comment_single_tag_or_none.sample(withReplacement=False, fraction=fraction * 1.1, seed=42).limit(100000)

# 4. Store the rest of the data (Not selected)
# We use a Left Anti Join to find rows in post_with_id that are NOT in sampling
rest_data = comment_single_tag_or_none.join(sampling, on="row_id", how="left_anti")

# Optional: Remove the row_id column if you don't need it anymore
sampling = sampling.drop("row_id")
rest_data = rest_data.drop("row_id")

# บน Dataproc ถ้าไม่ unpersist หน่วยความจำอาจจะเต็มสำหรับ Job ถัดไป
comment_single_tag_or_none.unpersist()

print(f"Sampling count: {sampling.count()}")
print(f"Rest data count: {rest_data.count()}")

Sampling count: 55216


Rest data count: 3739964


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit

# Define a window specification to generate a sequential number
window_spec = Window.orderBy(lit(1)) # Use a literal to create a single partition for ordering

# Add the new column with repeating sequence 1,2,3,4,5
sampling = sampling.withColumn("Subgroup", (row_number().over(window_spec) - 1) % 5 + 1)

sampling.show(5)

26/04/26 09:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 0

+--------------------+-------------------+------------+--------------------+--------------+---------+-----+---+-------+--------+
|                body|        created_utc|is_submitter|              author|removal_reason|subreddit|score|ups|AI_Name|Subgroup|
+--------------------+-------------------+------------+--------------------+--------------+---------+-----+---+-------+--------+
|they’re not the o...|2025-06-02 20:54:01|        true|Pleasant-Shallot-707|          NULL|  ChatGPT|    2|  2|ChatGPT|       1|
|looks like toto w...|2025-06-02 21:04:51|       false|          delboy8888|          NULL|  ChatGPT|    1|  1|ChatGPT|       2|
|i’m glad to hear ...|2025-06-02 21:06:45|       false|       onceandfloral|          NULL|  ChatGPT|    3|  3|ChatGPT|       3|
|that looks like a...|2025-06-02 21:09:29|       false|          RainoireVT|          NULL|  ChatGPT|    2|  2|ChatGPT|       4|
|\n\nthe ai is may...|2025-06-02 21:11:19|       false|        putsonshorts|          NULL|  Chat

## Save sampling and rest_data to Data Lake

In [ ]:
# Save File

target_path_parquet1 = "gs://reddit-ai-2/process_data/Sampling_Comment_for_LLM/sampling_comment_for_LLM.parquet"
sampling.write.mode("overwrite").parquet(target_path_parquet1)
print(f"บันทึกไฟล์ sampling_for_LLM.parquet เรียบร้อยแล้วที่ GCS")

target_path_parquet2 = "gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_single_tag_or_non.parquet"
rest_data.write.mode("overwrite").parquet(target_path_parquet2)
print(f"บันทึกไฟล์ comment_single_tag_or_non.parquet เรียบร้อยแล้วที่ GCS")

sampling.unpersist()
rest_data.unpersist()

26/04/26 09:58:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:58:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:58:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:58:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:58:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:58:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 0

บันทึกไฟล์ sampling_for_LLM.parquet เรียบร้อยแล้วที่ GCS


บันทึกไฟล์ comment_single_tag_or_non.parquet เรียบร้อยแล้วที่ GCS


DataFrame[body: string, created_utc: string, is_submitter: boolean, author: string, removal_reason: string, subreddit: string, score: bigint, ups: bigint, AI_Name: string]

In [ ]:
rest_data.show(3)

+--------------------+-------------------+------------+----------------+--------------+---------+-----+---+-------+
|                body|        created_utc|is_submitter|          author|removal_reason|subreddit|score|ups|AI_Name|
+--------------------+-------------------+------------+----------------+--------------+---------+-----+---+-------+
|                same|2025-06-02 20:41:51|       false|    thecrew2game|          NULL|  ChatGPT|    1|  1|ChatGPT|
|i’m impressed by ...|2025-06-02 20:42:14|       false|  Livid_Pace9787|          NULL|  ChatGPT|    2|  2|ChatGPT|
|this is my favori...|2025-06-02 20:42:28|       false|sexy_throwawayME|          NULL|  ChatGPT|    2|  2|ChatGPT|
+--------------------+-------------------+------------+----------------+--------------+---------+-----+---+-------+
only showing top 3 rows



In [ ]:
spark.stop()